In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


###　パッチを作る場合

In [2]:
!rm -rf /content/opticspy
!git clone --depth 1 https://github.com/Sterncat/opticspy.git /content/opticspy
%cd /content/opticspy


Cloning into '/content/opticspy'...
remote: Enumerating objects: 1974, done.
remote: Counting objects: 100% (1974/1974), done.
remote: Compressing objects: 100% (1094/1094), done.
remote: Total 1974 (delta 763), reused 1909 (delta 752), pack-reused 0 (from 0)
Receiving objects: 100% (1974/1974), 6.04 MiB | 16.15 MiB/s, done.
Resolving deltas: 100% (763/763), done.
/content/opticspy


In [3]:
import pathlib, re

REPO = pathlib.Path("/content/opticspy")
PKG  = REPO / "opticspy"

def patch_relative_imports():
    changed = 0
    for py in PKG.rglob("*.py"):
        txt = py.read_text(encoding="utf-8", errors="ignore").splitlines()
        out = []
        local_mods = {p.stem for p in py.parent.glob("*.py") if p.stem != "__init__"}

        for line in txt:
            m = re.match(r"^(\s*)import\s+([A-Za-z_]\w*)\s*$", line)
            if m:
                indent, mod = m.group(1), m.group(2)
                if mod in local_mods:
                    out.append(f"{indent}from . import {mod}")
                    changed += 1
                    continue

            m2 = re.match(r"^(\s*)from\s+([A-Za-z_]\w*)\s+import\s+(.*)$", line)
            if m2:
                indent, mod, rest = m2.group(1), m2.group(2), m2.group(3)
                if mod in local_mods:
                    out.append(f"{indent}from .{mod} import {rest}")
                    changed += 1
                    continue

            out.append(line)

        new = "\n".join(out) + "\n"
        old = py.read_text(encoding="utf-8", errors="ignore")
        if new != old:
            py.write_text(new, encoding="utf-8")

    # opticspy/__init__.py のトップレベル import 群も相対化
    init_py = PKG / "__init__.py"
    root_mods = {p.stem for p in PKG.glob("*.py") if p.stem != "__init__"}
    t = init_py.read_text(encoding="utf-8", errors="ignore")

    def repl_import_list(m):
        indent = m.group(1)
        mods = [x.strip() for x in m.group(2).split(",")]
        lines = []
        for mod in mods:
            if mod in root_mods:
                lines.append(f"{indent}from . import {mod}")
            else:
                lines.append(f"{indent}import {mod}")
        return "\n".join(lines)

    t2 = re.sub(r"(?m)^(\s*)import\s+([A-Za-z_]\w*(?:\s*,\s*[A-Za-z_]\w*)+)\s*$", repl_import_list, t)
    for mod in root_mods:
        t2 = re.sub(rf"(?m)^(\s*)import\s+{mod}\s*$", rf"\1from . import {mod}", t2)

    if t2 != t:
        init_py.write_text(t2, encoding="utf-8")
        changed += 1

    return changed

def patch_matplotlib_title():
    draw_py = PKG / "ray_tracing" / "draw.py"
    s = draw_py.read_text(encoding="utf-8", errors="ignore").splitlines()
    out = []
    changed = 0
    for line in s:
        m = re.match(r"^(\s*)fig\.canvas\.set_window_title\((.*)\)\s*$", line)
        if m:
            indent = m.group(1)
            out += [
                f"{indent}# Colab/Agg-safe window title (no-op if backend has no manager)",
                f"{indent}try:",
                f"{indent}    mng = getattr(fig.canvas, 'manager', None)",
                f"{indent}    if mng and hasattr(mng, 'set_window_title'):",
                f"{indent}        mng.set_window_title('View Lens')",
                f"{indent}except Exception:",
                f"{indent}    pass",
            ]
            changed += 1
        else:
            out.append(line)
    if changed:
        draw_py.write_text("\n".join(out) + "\n", encoding="utf-8")
    return changed

c1 = patch_relative_imports()
c2 = patch_matplotlib_title()
print("patched relative imports:", c1)
print("patched matplotlib title:", c2)


patched relative imports: 0
patched matplotlib title: 1


In [4]:
!mkdir -p /content/drive/MyDrive/colab_repos
!rm -rf /content/drive/MyDrive/colab_repos/opticspy_patched
!rsync -a --delete /content/opticspy/ /content/drive/MyDrive/colab_repos/opticspy_patched/
!echo "Saved to: /content/drive/MyDrive/colab_repos/opticspy_patched"


Saved to: /content/drive/MyDrive/colab_repos/opticspy_patched


###ここから


In [3]:
!pip -q install numpy scipy matplotlib sympy pyyaml


In [4]:
%cd /content/drive/MyDrive/colab_repos/opticspy_patched
!pip -q install -e .


/content/drive/MyDrive/colab_repos/opticspy_patched
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [6]:
!python /content/opticspy_example1_exact_input_with_abcd_report_fixed.py

Paraxial (ABCD) results for Example 1 (air -> air), using nd indices
Ray vector: [y, theta]^T, distances in mm, light travels +z (left to right).

System matrix to last refracting surface (just after surface 8):
  A=0.8559927500, B=33.25790648
  C=-0.009988699859, D=0.7801430026

System matrix to image plane (including surface8 -> image translation):
  A=1.0257077393e-03, B=100.0330188
  C=-0.009988699859, D=0.7801430026

Effective focal length (EFL):
  f' = -1/C = 100.11312924 mm

Principal planes (absolute z, with z(surface2 vertex)=0):
  H  position z(H)  = 22.01057199 mm
  H' position z(H') = 19.22116157 mm

Image plane check (prescription vs ABCD back focal plane):
  Prescribed image plane z (surface9) = 119.23160400 mm
  ABCD back focal plane z (z8 + BFL)  = 119.33429081 mm
  Difference (ABCD - prescribed)      = 1.02686811e-01 mm

NOTE: この差が 0 にならない場合、処方の像面距離(t8=85.593426)が
      ndのパラキシャル後焦点距離(BFL)と一致していない、という意味です。
      パラキシャル焦点に合わせたいなら、t8 を BFL に置き換えると一致します。
Add field angle:0

In [7]:
from opticspy_abcd_layout_report_general import SurfaceSpec, ReportConfig

def example1_config(out_png: str = "example1_layout.png") -> ReportConfig:
    # あなたが貼ってくれた「正しい Example1 入力」そのまま
    prescription = [
        SurfaceSpec(1, 10000000.0, 1000000.0, "air", False),
        SurfaceSpec(2, 41.15909,   6.097555,  "S-BSM18_ohara", False),
        SurfaceSpec(3, -957.83146, 9.349584,  "air", False),
        SurfaceSpec(4, -51.32104,  2.032518,  "N-SF2_schott", False),
        SurfaceSpec(5, 42.37768,   5.995929,  "air", False),
        SurfaceSpec(6, 10000000.0, 4.065037,  "air", True),   # STOP (STO=True)
        SurfaceSpec(7, 247.44562,  6.097555,  "S-BSM18_ohara", False),
        SurfaceSpec(8, -40.04016,  85.593426, "air", False),
        SurfaceSpec(9, 10000000.0, 0.0,       "air", False),  # image plane
    ]

    return ReportConfig(
        prescription=prescription,
        field_angles_deg=[0, 14, 20],
        wavelengths_nm=[587.6, 656.3, 486.1],  # d/C/F lines (nm); middle=nd用
        fno=5.0,
        start_surface=2,
        end_surface=8,     # 最後の屈折面
        image_surface=9,   # 処方像面（surface9頂点）で比較
        image_z_abs=None,  # こちらを使うなら image_surface を None に
        reference_surface=2,  # z(surface2 vertex)=0
        draw_layout=True,
        out_png=out_png,
        opticspy_root=None,   # 必要ならパスを入れる or 環境変数 OPTICSPY_ROOT
    )


In [8]:
from opticspy_abcd_layout_report_general import run_report_and_optional_plot
cfg = example1_config(out_png="example1.png")
run_report_and_optional_plot(cfg)


Paraxial (ABCD) results (air -> air), using middle-wavelength indices
Ray vector: [y, theta]^T, distances in mm, light travels +z (left to right).

System matrix to last refracting surface (just after surface 8):
  A=0.8559927500, B=33.25790648
  C=-0.009988699859, D=0.7801430026

System matrix to image plane (including end_surface -> image translation):
  A=1.0257077393e-03, B=100.0330188
  C=-0.009988699859, D=0.7801430026

Effective focal length (EFL):
  f' = -1/C = 100.11312924 mm

Principal planes (absolute z, with z(surface2 vertex)=0):
  H  position z(H)  = 22.01057199 mm
  H' position z(H') = 19.22116157 mm

Image plane check (prescription vs ABCD back focal plane):
  Prescribed image plane z = 119.23160400 mm
  Difference (ABCD - prescribed)      = 1.02686811e-01 mm
  ABCD back focal plane z  = 119.33429081 mm

NOTE: 差が 0 にならない場合、処方の像面位置がパラキシャル後焦点面(BFL)と
      一致していないことを意味します。像面をパラキシャル焦点に合わせたいなら
      像面位置を z_end + BFL に設定してください。
Add field angle:0.0 degree done
Add field angle

{'ABCD': {'A': 0.8559927499805775,
  'B': 33.25790648228942,
  'C': -0.00998869985927796,
  'D': 0.7801430026313055},
 'cardinals': {'fprime': 100.11312924485907,
  'h': 22.01057199295875,
  'hprime': 14.417016433391177,
  'BFL': 85.6961128114679,
  'FFL': 78.10255725190032},
 'z': {2: 0.0,
  3: 6.097555,
  4: 15.447139,
  5: 17.479657,
  6: 23.475586,
  7: 27.540623,
  8: 33.638177999999996,
  9: 119.23160399999999},
 'zH': 22.01057199295875,
 'zHp': 19.22116156660882,
 'z_end': 33.638177999999996,
 'z_img_prescribed': 119.23160399999999,
 'z_img_abcd': 119.3342908114679,
 'M_to_img': array([[ 1.02570774e-03,  1.00033019e+02],
        [-9.98869986e-03,  7.80143003e-01]]),
 'cfg': ReportConfig(prescription=[SurfaceSpec(num=1, R=10000000.0, t=1000000.0, glass='air', stop=False), SurfaceSpec(num=2, R=41.15909, t=6.097555, glass='S-BSM18_ohara', stop=False), SurfaceSpec(num=3, R=-957.83146, t=9.349584, glass='air', stop=False), SurfaceSpec(num=4, R=-51.32104, t=2.032518, glass='N-SF2_scho